# **Prepare Data**

In [ ]:
# !pip -q install torch torchvision transformers timm

In [ ]:
import os
import pandas as pd
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from timm import create_model

In [ ]:
class HouseDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.data = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.data.iloc[idx, 0])
        image = Image.open(img_path).convert('RGB')
        label = torch.tensor(self.data.iloc[idx, 1], dtype=torch.float32)

        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

In [ ]:
train_dataset = HouseDataset("/kaggle/input/competitions/super-ai-engineer-season-6-individual-hackathon-house-recognition/train.csv", "/kaggle/input/competitions/super-ai-engineer-season-6-individual-hackathon-house-recognition/train/train", transform=transform)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

# **Model**

In [ ]:
ckpt = torch.load("/kaggle/working/convnextv2_base_house_model.pth", map_location=device)
# model = create_model("convnextv2_tiny", pretrained=True, num_classes=1)
model = create_model("convnextv2_base", pretrained=True, num_classes=1)
model.load_state_dict(ckpt)
model = model.to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))
model.train()

ConvNeXt(
  (stem): Sequential(
    (0): Conv2d(3, 128, kernel_size=(4, 4), stride=(4, 4))
    (1): LayerNorm2d((128,), eps=1e-06, elementwise_affine=True)
  )
  (stages): Sequential(
    (0): ConvNeXtStage(
      (downsample): Identity()
      (blocks): Sequential(
        (0): ConvNeXtBlock(
          (conv_dw): Conv2d(128, 128, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=128)
          (norm): LayerNorm((128,), eps=1e-06, elementwise_affine=True)
          (mlp): GlobalResponseNormMlp(
            (fc1): Linear(in_features=128, out_features=512, bias=True)
            (act): GELU()
            (drop1): Dropout(p=0.0, inplace=False)
            (grn): GlobalResponseNorm()
            (fc2): Linear(in_features=512, out_features=128, bias=True)
            (drop2): Dropout(p=0.0, inplace=False)
          )
          (shortcut): Identity()
          (drop_path): Identity()
        )
        (1): ConvNeXtBlock(
          (conv_dw): Conv2d(128, 128, kernel_size=(7, 7), strid

In [ ]:
# # model = create_model("convnextv2_tiny", pretrained=True, num_classes=1)
# model = create_model("convnextv2_base", pretrained=True, num_classes=1)
# model = model.to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
def train_model(model, dataloader, criterion, optimizer, epochs):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.train()

    for epoch in range(epochs):
        running_loss = 0.0
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device).unsqueeze(1)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        print(f"Epoch {epoch+1}, Loss: {running_loss/len(dataloader):.4f}")

In [ ]:
train_model(model, train_loader, criterion, optimizer, epochs=5)

Epoch 1, Loss: 0.0475
Epoch 2, Loss: 0.0436
Epoch 3, Loss: 0.0393
Epoch 4, Loss: 0.0298
Epoch 5, Loss: 0.0313


In [ ]:
torch.save(model.state_dict(), "convnextv2_base_house_model.pth")

Test folder image preprocessing

# **Prediction**

In [ ]:
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

In [ ]:
import torch
import os
import pandas as pd
from PIL import Image
from torchvision import transforms

In [ ]:
def predict(model, test_folder, output_csv):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()

    test_images = [f for f in os.listdir(test_folder) if f.lower().endswith(('png', 'jpg', 'jpeg'))]
    results = []

    with torch.no_grad():
        for img_name in test_images:
            img_path = os.path.join(test_folder, img_name)
            image = Image.open(img_path).convert('RGB')
            image = test_transform(image).unsqueeze(0).to(device)

            output = model(image)
            if isinstance(output, torch.Tensor):
                output = output.item()

            pred = 1 if output > 0 else 0
            results.append([os.path.splitext(img_name)[0], pred])

    df = pd.DataFrame(results, columns=["id", "answer"])
    df.to_csv(output_csv, index=False)
    print(f"Predictions saved to {output_csv}")
    print(df.head())

In [ ]:
predict(model, "/kaggle/input/competitions/super-ai-engineer-season-6-individual-hackathon-house-recognition/test/test", "submission.csv")

Predictions saved to submission.csv
         id  answer
0  fd0900c0       1
1  b17ced93       0
2  5157729e       1
3  9b25615e       0
4  1df1da60       0
